# 14 · Muestra para KPIs de integración y triaje (Sprint 4)
Genera `results/kpi_muestra.csv` (200 correos del **test held-out**, nunca vistos en entrenamiento):
150 en inglés (60 phishing · 45 spam · 45 legítimo) y 50 en español nativo (25 phishing · 25 legítimo; no existe spam nativo).
El texto se guarda **defangeado**; los scripts `lote_kpi.py` y `triaje_cronometro.py` refangean solo en memoria.
Descarga el CSV y cópialo a `deploy/pruebas/` en el equipo del stack. Solo necesita CPU.

In [1]:
# Celda única: muestra estratificada del test held-out
import os, sys, json
from pathlib import Path
import pandas as pd, pyarrow.parquet as pq
EN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
if EN_COLAB:
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
else:                                                    # Kaggle: dataset con la carpeta PI_G17
    BASE = next(Path("/kaggle/input").rglob("PI_G17"))
sys.path.insert(0, str(BASE / "src")); import ioc_utils as ioc
PROC, RES = BASE / "data/processed", BASE / "results"; RES.mkdir(exist_ok=True)
SEMILLA = 17

def cols(p): return set(pq.read_schema(p).names)
# Inglés: test held-out de emails_v1 + texto defangeado de emails_v0 (mismo cruce que el notebook 13)
V1 = pd.read_parquet(PROC / "emails_v1.parquet", columns=["id", "hash", "clase", "split"])
V0 = pd.read_parquet(BASE / "data/interim/emails_v0.parquet", columns=["hash", "texto_defang"])
EN = V1[V1.split == "test"].merge(V0.drop_duplicates("hash"), on="hash")
cuota_en = {"phishing": 60, "spam": 45, "legitimo": 45}
EN = pd.concat([EN[EN.clase == c].sample(n, random_state=SEMILLA) for c, n in cuota_en.items()])
EN = EN.assign(idioma="en")[["id", "idioma", "clase", "texto_defang"]]
# Español: solo corpus NATIVOS (regla D18: el traducido nunca se usa para evaluar)
p_es = PROC / "emails_es_v2.parquet"
col_txt = "texto_defang" if "texto_defang" in cols(p_es) else "texto_modelo"
ES = pd.read_parquet(p_es, columns=["id", "grupo", "clase", "split", col_txt]).rename(columns={col_txt: "texto_defang"})
ES = ES[ES.grupo.isin(["SpaPhish", "SpearPhishMX"]) & (ES.split == "test")]
ES = pd.concat([ES[ES.clase == c].sample(min(25, (ES.clase == c).sum()), random_state=SEMILLA) for c in ("phishing", "legitimo")])
ES = ES.assign(idioma="es")[["id", "idioma", "clase", "texto_defang"]]
M = pd.concat([EN, ES]).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
# KPI de defanging: el CSV no puede contener IOCs vivos
vivos = int(M.texto_defang.map(lambda t: bool(ioc.iocs_vivos(t))).sum())
assert vivos == 0, f"{vivos} filas con IOC vivo: NO exportar"
M.to_csv(RES / "kpi_muestra.csv", index=False, encoding="utf-8")
RESUMEN = {"archivo": str(RES / "kpi_muestra.csv"), "filas": len(M), "texto_es": col_txt,
           "por_idioma_clase": M.groupby(["idioma", "clase"]).size().to_dict(),
           "con_url_defangeada_pct": round(100 * M.texto_defang.str.contains(r"hxxp|\[\.\]", regex=True).mean(), 1),
           "ioc_vivos": vivos}
print("===== RESUMEN (14) =====")
print(json.dumps({k: (str(v) if isinstance(v, dict) else v) for k, v in RESUMEN.items()}, ensure_ascii=False))


Mounted at /content/drive
===== RESUMEN (14) =====
{"archivo": "/content/drive/MyDrive/PI_G17/results/kpi_muestra.csv", "filas": 200, "texto_es": "texto_modelo", "por_idioma_clase": "{('en', 'legitimo'): 45, ('en', 'phishing'): 60, ('en', 'spam'): 45, ('es', 'legitimo'): 25, ('es', 'phishing'): 25}", "con_url_defangeada_pct": 52.0, "ioc_vivos": 0}
